In [ ]:
# Install Ollama Python client library
pip install ollama

Defaulting to user installation because normal site-packages is not writeable
You should consider upgrading via the '/Library/Developer/CommandLineTools/usr/bin/python3 -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.


In [ ]:
"""
================================
OLLAMA RAG System - Imports
================================
Import necessary libraries for building a Retrieval Augmented Generation (RAG) system
using Ollama local LLM with vector similarity search and reranking.

Libraries used:
- ollama: Local LLM inference
- PyPDF2: PDF text extraction
- chromadb: Vector database for semantic search
- dotenv: Environment variable management
"""

import os
from ollama import chat
from ollama import ChatResponse
from dotenv import load_dotenv
import PyPDF2
import chromadb

In [ ]:
"""
================================
PDF Loading Function
================================
Extract text content from PDF files for RAG processing.
"""

def load_pdf(path):
    """
    Extract all text from a PDF file.
    
    Args:
        path (str): File path to the PDF document
        
    Returns:
        str: Concatenated text content from all PDF pages
    
    Example:
        >>> text = load_pdf("document.pdf")
    """
    text = ""
    with open(path, "rb") as f:
        reader = PyPDF2.PdfReader(f)
        for page in reader.pages:
            text += page.extract_text() + "\n"
    return text

# Load F1 rules PDF for Q&A system
pdf_text = load_pdf("/Users/rush2hemant/Documents/Python DS/Docs/F1rules.pdf")

In [ ]:
"""
================================
Text Chunking for Semantic Search
================================
Split the PDF text into overlapping chunks for better semantic understanding
and vector embedding. The overlap preserves context across chunk boundaries.
"""

from langchain_text_splitters import RecursiveCharacterTextSplitter

# Configure text splitter with overlap for context preservation
splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,    # Size of each text chunk (characters)
    chunk_overlap=200,  # Overlap between chunks to maintain context continuity
)

# Split the loaded PDF text into chunks
chunks = splitter.split_text(pdf_text)
print(f"Loaded {len(chunks)} chunks")

Loaded 442 chunks


In [ ]:
"""
================================
Vector Embeddings & Storage
================================
Generate dense vector embeddings for each chunk using a pre-trained model
and store them in a persistent vector database (ChromaDB) for efficient
semantic similarity search.

Model: all-MiniLM-L6-v2 (384-dimensional embeddings)
Database: ChromaDB (persistent local storage)
"""

from sentence_transformers import SentenceTransformer

# Load pre-trained embedding model (384-dimensional, good for semantic search)
embed_model = SentenceTransformer("all-MiniLM-L6-v2")

# Initialize persistent ChromaDB client for storing embeddings
# Data persists in 'chroma_store' directory across sessions
chroma_client = chromadb.PersistentClient(path="chroma_store")
collection = chroma_client.get_or_create_collection("f1_rules_ol")

# Generate vector embeddings for all document chunks
embeddings = embed_model.encode(chunks).tolist()

# Store chunks with their embeddings in the vector database
# Each chunk is indexed by a unique ID (chunk_0, chunk_1, etc.)
collection.add(
    documents=chunks,
    embeddings=embeddings,
    ids=[f"chunk_{i}" for i in range(len(chunks))]
)

print("Successfully stored embeddings in vector database!")

Stored in vector DB!


In [ ]:
"""
================================
Semantic Similarity Retrieval
================================
Retrieve the most semantically similar chunks from the vector database
using the query embedding. This stage fetches more results than needed
for subsequent reranking.
"""

# User query about F1 rules
# Uncomment the line below for interactive input:
## query = input("What do you want to know about F1 rules: ")

query = "Can a team pit when pit lane is closed during safety car"

# Generate embedding for the user query using the same model
query_embedding = embed_model.encode([query]).tolist()

# Initial retrieval: fetch top 10 semantically similar chunks
# These will be reranked in the next step to improve relevance
results_top10 = collection.query(
    query_embeddings=query_embedding,
    n_results=10
)

# Extract the chunks from query results
initial_chunks = results_top10["documents"][0]

In [ ]:
"""
================================
Reranking with Cross-Encoder
================================
Rerank the retrieved chunks using a more sophisticated cross-encoder model
to improve relevance. The cross-encoder directly compares query-document pairs
for more accurate ranking than semantic similarity alone.

Model: BAAI/bge-reranker-v2-m3 (specifically trained for reranking)
"""

from sentence_transformers import CrossEncoder

# Load cross-encoder model optimized for query-document relevance scoring
model = CrossEncoder("BAAI/bge-reranker-v2-m3")

# Refine the query for better results
query = "What are the rules for overtaking under yellow flags?"

# Create query-document pairs for reranking
pairs = [[query, d] for d in initial_chunks]

# Score each query-document pair (higher score = more relevant)
scores = model.predict(pairs)

# Sort chunks by relevance score in descending order
ranked_chunks = sorted(zip(initial_chunks, scores), key=lambda x: x[1], reverse=True)

# Extract top 3 most relevant chunks for the LLM context
top_chunks = ranked_chunks[:3]

In [ ]:
"""
================================
Prepare Context for LLM
================================
Extract and display the top reranked chunks that will be used as context
for the language model to generate accurate answers.
"""

# Extract documents from reranked chunks
final_chunks = []
for docs in top_chunks:
    print(docs[0])  # Display each chunk
    final_chunks.append(docs[0])

print(f"\n✓ Selected {len(final_chunks)} chunks for context")

is given taking all equipment with them. If any team personnel are touching an F1 Car or team equipment is connected to an F1 Car in the Fast Lane after the ﬁfteen (15) second signal has been shown, the driver of the F1 Car concerned must enter the Pit Lane following the laps behind the Safety Car described in Article B5.15.2(f), and may join the TTCS  from the end of the Pit Lane, once the Pit Lane exit is opened after the standing or rolling start resumption. A Stop-and-Go Penalty will be imposed on any driver who fails to enter the Pit Lane and resume from the Pit Lane. If any driver needs assistance after the ﬁfteen (15) second signal, they must raise their arm and, when the remainder of the F1 Cars able to do so have left the Pit Lane, marshals will be instructed to push the F1 Car into the Inner Lane. In this case, marshals with yellow ﬂags will stand beside any F1 Car concerned to warn drivers behind. f. At the resumption time the pit exit will be opened, all cars should leave
a

In [ ]:
"""
================================
LLM Inference with Ollama
================================
Generate a response using a local Ollama LLM model with the retrieved
context as reference. This is the final step of the RAG pipeline.

Model: gemma3:4b (lightweight, runs locally)
"""

# Combine the top reranked chunks into a single context string
context = "\n\n".join(final_chunks)

# Build a detailed prompt with context and instructions for the LLM
user_prompt = f"""
Use the following context to answer the question. Assume you are a formula1 expert.

Context:
{context}

Question:
{query}

Answer:
"""

# Call Ollama LLM to generate response based on the context
# stream=False means we wait for the complete response before proceeding
ol_chat = chat(
    model='gemma3:4b',  # Local model running via Ollama
    messages=[{'role': 'user', 'content': user_prompt}],
    stream=False,
)

In [ ]:
"""
================================
Display Final Answer
================================
Print the generated response from the Ollama LLM.
"""

print("=" * 60)
print("FINAL ANSWER:")
print("=" * 60)
print(ol_chat.message.content)

Okay, let’s break down the overtaking rules specifically related to yellow flags, based on this excerpt from the 2026 F1 Sporting Regulations.

The key section here is **f.** which states: “marshals will be instructed to push the F1 Car into the Inner Lane. In this case, marshals with yellow ﬂags will stand beside any F1 Car concerned to warn drivers behind.”

**Essentially, when a car is being pushed into the Inner Lane due to a problem (indicated by yellow flags), drivers MUST slow down significantly and avoid overtaking.** It's a clear warning signal that there's a hazard ahead and the track is temporarily compromised.

**Important takeaway:** The presence of yellow flags and marshals signifies a reduced-speed zone. Overtaking is strictly prohibited in this situation. Drivers need to be extra cautious and aware of the situation.

Do you want me to elaborate on any specific aspect of this, such as the potential consequences of failing to respect the yellow flag situation, or perhaps 